# E3 · LLM chưa thấy · Chủ: Người 1 · Phục vụ Chương 5 (mục 5.3, RQ2)
Huấn luyện chỉ trên văn bản người + LLM cũ (`config.OLD_GENERATORS`), chọn ngưỡng trên validation (cũng chỉ người + LLM cũ),
đo trên test theo từng LLM → **T5.5**.

In [ ]:
# === BOOTSTRAP: giữ nguyên ở đầu mọi notebook ===
OWNER = "ten-cua-ban"     # điền tên để log kết quả
REPO  = "https://github.com/<ten-tai-khoan>/ai-text-detection.git"   # sửa thành repo của nhóm

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    url = REPO
    try:   # repo private: lưu Personal Access Token vào Colab Secrets với tên GH_TOKEN
        from google.colab import userdata
        url = REPO.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
sys.path.insert(0, "/content/repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/repo/requirements.txt"])

from aitext import config, spark_utils
spark_utils.ensure_pyspark()
spark = spark_utils.get_spark()
config.ensure_dirs()
import time, pandas as pd, matplotlib.pyplot as plt
from pyspark.sql import functions as F
print("Drive:", config.DRIVE_ROOT)

In [ ]:
from aitext import data, models, evaluate
train, val, test = data.load_splits(spark)
old = (F.col("label") == 0) | F.col("model").isin(config.OLD_GENERATORS)
tr = data.with_class_weight(train.filter(old)).cache()
m = models.fit(tr, defended=True)                         # dùng regParam đã chốt ở E1
thr = evaluate.thr_at_fpr(evaluate.score(m, val.filter(old), models.text_col(True)))
pred = evaluate.score(m, test, models.text_col(True)).cache()
human = pred.filter("label = 0"); rows = []
for g_ in config.OLD_GENERATORS + config.NEW_GENERATORS:
    sub = human.unionByName(pred.filter(F.col("model") == g_))
    rows.append({"LLM": g_, "nhóm": "đã thấy" if g_ in config.OLD_GENERATORS else "CHƯA thấy",
                 "AUROC": evaluate.auroc(sub), "TPR": evaluate.rate_above(sub, 1, thr)})
t55 = pd.DataFrame(rows); display(t55.round(3))
evaluate.save_table(t55, "T5.5_unseen_generators")
display(t55.groupby("nhóm")[["AUROC", "TPR"]].mean().round(3))

## Ghi chú cho Chương 5
- ...